In [ ]:
# Example 1: Import the main client classes and verify they are constructible types.
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.aio import ImageAnalysisClient as AsyncImageAnalysisClient

assert isinstance(ImageAnalysisClient, type), "ImageAnalysisClient should be a class"
assert isinstance(AsyncImageAnalysisClient, type), "Async ImageAnalysisClient should be a class"

print(f"Sync client: {ImageAnalysisClient.__module__}.{ImageAnalysisClient.__name__}")
print(f"Async client: {AsyncImageAnalysisClient.__module__}.{AsyncImageAnalysisClient.__name__}")

# Verify primary method exists on the sync client.
assert hasattr(ImageAnalysisClient, 'analyze'), "Client should have an analyze method"
assert hasattr(ImageAnalysisClient, 'analyze_from_url'), "Client should have analyze_from_url method"
print("Client analyze methods present")

In [ ]:
# Example 2: Inspect the VisualFeatures enum (primary input parameter to analyze).
from azure.ai.vision.imageanalysis.models import VisualFeatures

# VisualFeatures is a string enum listing image analysis capabilities.
expected_features = {'TAGS', 'OBJECTS', 'CAPTION', 'DENSE_CAPTIONS', 'READ', 'SMART_CROPS', 'PEOPLE'}
available = {f.name for f in VisualFeatures}
print(f"Available VisualFeatures: {sorted(available)}")

# At least the core capabilities should be present.
missing = expected_features - available
assert not missing, f"Missing expected features: {missing}"

# Values should be strings (per the REST contract).
assert all(isinstance(f.value, str) for f in VisualFeatures)
print(f"All {len(available)} features have string values")

In [ ]:
# Example 3: Construct the client with a fake credential (no network call) and check shape.
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.core.credentials import AzureKeyCredential

# We don't actually call the service; we just verify the constructor accepts
# the documented (endpoint, credential) signature without raising.
client = ImageAnalysisClient(
    endpoint="https://example.cognitiveservices.azure.com/",
    credential=AzureKeyCredential("fake-key-for-construction-test"),
)

assert client is not None
# The client exposes an endpoint via the underlying pipeline config.
assert hasattr(client, '_config') or hasattr(client, '_client')
print(f"Constructed client: {type(client).__name__}")

# Clean up.
client.close()
print("Client closed cleanly")

In [ ]:
# Example 4: Inspect the response model classes (output of analyze()).
from azure.ai.vision.imageanalysis import models as ia_models

# The library exposes a rich set of response models.
expected_models = [
    'ImageAnalysisResult',
    'CaptionResult',
    'DenseCaptionsResult',
    'ObjectsResult',
    'PeopleResult',
    'ReadResult',
    'SmartCropsResult',
    'TagsResult',
    'ImageBoundingBox',
]

found = []
for name in expected_models:
    cls = getattr(ia_models, name, None)
    assert cls is not None, f"Expected model {name} missing"
    assert isinstance(cls, type)
    found.append(name)

print(f"Found {len(found)} response models: {found}")

In [ ]:
# Example 5: Build an ImageBoundingBox model instance and round-trip its fields.
from azure.ai.vision.imageanalysis.models import ImageBoundingBox

# Python attribute names are width/height; the REST wire names are 'w'/'h'.
bbox = ImageBoundingBox(x=10, y=20, width=100, height=50)

assert bbox.x == 10
assert bbox.y == 20
assert bbox.width == 100
assert bbox.height == 50

# as_dict() serializes with the REST wire names (azure-core convention).
as_dict = bbox.as_dict()
assert as_dict.get('x') == 10
assert as_dict.get('w') == 100
assert as_dict.get('h') == 50

print(f"ImageBoundingBox round-trip OK: {as_dict}")
print("All azure.ai.vision.imageanalysis smoke tests passed")
